# Fine-tuning IndoBERT untuk Klasifikasi Pantun

Notebook ini dirancang untuk dijalankan di Google Colab (via plugin Colab di VS Code).

Fitur utama:
- Membaca dataset `Dataset_Final_Klasifikasi.csv`
- Split stratified train/valid/test
- Menangani data imbalanced dengan class-weighted loss
- Melatih model IndoBERT untuk klasifikasi multi-kelas
- Mencatat waktu komputasi training
- Menyimpan model dan hasil evaluasi ke Google Drive
- Membuat file `.zip` agar mudah diunduh

In [ ]:
# Jika di Colab, jalankan sekali untuk install dependensi.
# Kalau sudah terpasang, bisa dilewati.

!pip -q install -U transformers datasets evaluate accelerate scikit-learn pandas numpy seaborn

In [ ]:
import os
import json
import time
import random
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Konfigurasi path input/output.
# Untuk Colab, notebook ini otomatis mount Google Drive.

IN_COLAB = False
try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/pantun_article3_indobert')
    DATA_PATH = Path('/content/drive/MyDrive/pantun_article3_indobert/Dataset_Final_Klasifikasi.csv')
    # Jika dataset belum ada di folder drive target, fallback ke path lokal Colab jika diunggah manual.
    if not DATA_PATH.exists():
        DATA_PATH = Path('/content/Dataset_Final_Klasifikasi.csv')
else:
    BASE_DIR = Path('./outputs/article3/indobert')
    DATA_PATH = Path('../../data/article3/Dataset_Final_Klasifikasi.csv')

RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / f'run_{RUN_TS}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('DATA_PATH:', DATA_PATH)
print('RUN_DIR:', RUN_DIR)

In [ ]:
# Load dataset dan validasi kolom.

df = pd.read_csv(DATA_PATH, sep=';', engine='python', on_bad_lines='skip')
required_cols = ['text_pantun', 'Kategori Pakar']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f'Kolom wajib tidak ditemukan: {c}')

# Bersihkan data.
df = df[required_cols].copy()
df['text_pantun'] = df['text_pantun'].astype(str).str.strip()
df['Kategori Pakar'] = df['Kategori Pakar'].astype(str).str.strip()
df = df[(df['text_pantun'] != '') & (df['Kategori Pakar'] != '')]
df = df.reset_index(drop=True)

print('Jumlah data setelah cleaning:', len(df))
print('\nDistribusi label:')
print(df['Kategori Pakar'].value_counts())

In [ ]:
# Encoding label dan split stratified train/valid/test.

label_list = sorted(df['Kategori Pakar'].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

df['label'] = df['Kategori Pakar'].map(label2id)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df['label']
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df['label']
)

print('Train:', train_df.shape, 'Valid:', valid_df.shape, 'Test:', test_df.shape)
print('\nDistribusi train:')
print(train_df['Kategori Pakar'].value_counts())

# Simpan metadata split untuk reproducibility.
split_meta = {
    'seed': SEED,
    'n_total': int(len(df)),
    'n_train': int(len(train_df)),
    'n_valid': int(len(valid_df)),
    'n_test': int(len(test_df)),
    'labels': label_list,
}
(Path(RUN_DIR) / 'split_metadata.json').write_text(json.dumps(split_meta, indent=2, ensure_ascii=False), encoding='utf-8')

In [ ]:
# Siapkan dataset Hugging Face + tokenizer.

MODEL_NAME = 'indobenchmark/indobert-base-p1'
MAX_LENGTH = 256

train_ds = Dataset.from_pandas(train_df[['text_pantun', 'label']], preserve_index=False)
valid_ds = Dataset.from_pandas(valid_df[['text_pantun', 'label']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['text_pantun', 'label']], preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['text_pantun'],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(valid_ds)
print(test_ds)

In [ ]:
# Penanganan imbalance: class-weighted loss.

class_counts = train_df['label'].value_counts().sort_index()
num_classes = len(label_list)
num_train = len(train_df)

# Formula: w_c = N / (K * n_c)
class_weights = torch.tensor(
    [num_train / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float
)

print('Class weights:')
for i, w in enumerate(class_weights.tolist()):
    print(f'{id2label[i]}: {w:.4f}')

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Siapkan model + metric.

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id,
)

metric_accuracy = evaluate.load('accuracy')
metric_f1 = evaluate.load('f1')
metric_precision = evaluate.load('precision')
metric_recall = evaluate.load('recall')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = metric_accuracy.compute(predictions=preds, references=labels)['accuracy']
    f1_macro = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    f1_weighted = metric_f1.compute(predictions=preds, references=labels, average='weighted')['f1']
    precision_macro = metric_precision.compute(predictions=preds, references=labels, average='macro')['precision']
    recall_macro = metric_recall.compute(predictions=preds, references=labels, average='macro')['recall']

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

In [ ]:
# Hyperparameter training (silakan sesuaikan jika GPU kecil).

OUTPUT_DIR = RUN_DIR / 'model'
LOG_DIR = RUN_DIR / 'logs'

# Checkpoint berbasis step untuk antisipasi timeout Colab.
SAVE_STEPS = 200
EVAL_STEPS = 200

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=True,
    evaluation_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=3,
    report_to='none',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

In [ ]:
# Cek checkpoint yang sudah ada sebelum training.

checkpoint_dirs = sorted(
    [p for p in OUTPUT_DIR.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1])
)

print('OUTPUT_DIR:', OUTPUT_DIR)
print('Jumlah checkpoint ditemukan:', len(checkpoint_dirs))

if checkpoint_dirs:
    print('Daftar checkpoint:')
    for p in checkpoint_dirs:
        print('-', p)
    print('Checkpoint terbaru:', checkpoint_dirs[-1])
else:
    print('Belum ada checkpoint. Training akan mulai dari awal.')

print('save_steps =', SAVE_STEPS, '| eval_steps =', EVAL_STEPS)

In [ ]:
# Opsional: bersihkan checkpoint lama agar storage Google Drive tidak cepat penuh.
# Ubah PRUNE_OLD_CHECKPOINTS=True jika ingin menghapus checkpoint lama.

PRUNE_OLD_CHECKPOINTS = False
KEEP_LAST_N = 2

checkpoint_dirs = sorted(
    [p for p in OUTPUT_DIR.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1])
)

if not checkpoint_dirs:
    print('Tidak ada checkpoint untuk dibersihkan.')
elif not PRUNE_OLD_CHECKPOINTS:
    print('Mode aman aktif: tidak ada checkpoint yang dihapus.')
    print('Set PRUNE_OLD_CHECKPOINTS=True untuk mengaktifkan cleanup.')
    print('Jumlah checkpoint saat ini:', len(checkpoint_dirs))
else:
    to_delete = checkpoint_dirs[:-KEEP_LAST_N] if len(checkpoint_dirs) > KEEP_LAST_N else []
    if not to_delete:
        print(f'Checkpoint <= {KEEP_LAST_N}, tidak ada yang dihapus.')
    else:
        deleted = 0
        for ckpt in to_delete:
            for fp in sorted(ckpt.rglob('*'), reverse=True):
                if fp.is_file() or fp.is_symlink():
                    fp.unlink(missing_ok=True)
                elif fp.is_dir():
                    fp.rmdir()
            ckpt.rmdir()
            deleted += 1
        print(f'Selesai: {deleted} checkpoint lama dihapus, menyisakan {KEEP_LAST_N} terbaru.')

checkpoint_dirs_after = sorted(
    [p for p in OUTPUT_DIR.glob('checkpoint-*') if p.is_dir()],
    key=lambda p: int(p.name.split('-')[-1])
)
print('Checkpoint tersisa:', len(checkpoint_dirs_after))

In [ ]:
# Train + catat waktu komputasi.
# Jika ada checkpoint sebelumnya di OUTPUT_DIR, training akan dilanjutkan otomatis.

checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
resume_ckpt = str(checkpoints[-1]) if checkpoints else None
if resume_ckpt:
    print('Melanjutkan dari checkpoint:', resume_ckpt)
else:
    print('Tidak ada checkpoint lama. Training dimulai dari awal.')

start_time = time.perf_counter()
train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
end_time = time.perf_counter()

training_time_seconds = end_time - start_time
training_time_minutes = training_time_seconds / 60

print(f'Training selesai dalam {training_time_seconds:.2f} detik ({training_time_minutes:.2f} menit).')

# Simpan log waktu training.
training_time_data = {
    'training_time_seconds': training_time_seconds,
    'training_time_minutes': training_time_minutes,
    'train_samples': len(train_df),
    'valid_samples': len(valid_df),
    'test_samples': len(test_df),
    'timestamp': RUN_TS,
    'resume_from_checkpoint': resume_ckpt,
}
(RUN_DIR / 'training_time.json').write_text(json.dumps(training_time_data, indent=2), encoding='utf-8')

In [ ]:
# Evaluasi di test set.

test_output = trainer.predict(test_ds)
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=-1)

acc = accuracy_score(y_true, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print('Test Accuracy      :', round(acc, 4))
print('Test Macro F1      :', round(f1_macro, 4))
print('Test Weighted F1   :', round(f1_weighted, 4))
print('Test Macro Precision:', round(precision_macro, 4))
print('Test Macro Recall   :', round(recall_macro, 4))

report = classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(num_classes)],
    output_dict=True,
    zero_division=0,
)

results = {
    'accuracy': float(acc),
    'f1_macro': float(f1_macro),
    'f1_weighted': float(f1_weighted),
    'precision_macro': float(precision_macro),
    'recall_macro': float(recall_macro),
    'training_time_seconds': float(training_time_seconds),
    'training_time_minutes': float(training_time_minutes),
}

(RUN_DIR / 'test_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
(RUN_DIR / 'classification_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

pd.DataFrame([results]).to_csv(RUN_DIR / 'test_metrics.csv', index=False)
print('Metrik test tersimpan.')

In [ ]:
# Confusion matrix.

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[id2label[i] for i in range(num_classes)],
    columns=[id2label[i] for i in range(num_classes)],
)

cm_df.to_csv(RUN_DIR / 'confusion_matrix.csv', index=True)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - IndoBERT (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# Simpan model terbaik dan tokenizer.

best_model_dir = RUN_DIR / 'best_model'
best_model_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(best_model_dir))
tokenizer.save_pretrained(str(best_model_dir))

# Simpan info konfigurasi training.
config_dump = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_classes': num_classes,
    'class_weights': [float(x) for x in class_weights.tolist()],
    'training_args': {
        'learning_rate': training_args.learning_rate,
        'per_device_train_batch_size': training_args.per_device_train_batch_size,
        'per_device_eval_batch_size': training_args.per_device_eval_batch_size,
        'num_train_epochs': training_args.num_train_epochs,
        'weight_decay': training_args.weight_decay,
    },
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(config_dump, indent=2), encoding='utf-8')

print('Model & tokenizer tersimpan di:', best_model_dir)

In [ ]:
# Kompres hasil run agar mudah diunduh.

zip_path = BASE_DIR / f'indobert_run_{RUN_TS}.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files_in in os.walk(RUN_DIR):
        for fname in files_in:
            full_path = Path(root) / fname
            arcname = full_path.relative_to(BASE_DIR)
            zf.write(full_path, arcname)

print('ZIP tersimpan:', zip_path)
print('Folder run   :', RUN_DIR)

if IN_COLAB:
    print('Anda bisa download ZIP langsung dari cell berikut (opsional).')

In [ ]:
# Opsional: download ZIP langsung dari Colab.
# Jalankan hanya jika IN_COLAB=True.

if IN_COLAB:
    files.download(str(zip_path))
else:
    print('Bukan environment Colab. File ZIP ada di:', zip_path)